# CIFAR-10 Image Classification Model

This notebook loads and explores CIFAR-10, trains a compact convolutional neural network, evaluates it, and saves the Keras model and class labels for the FastAPI inference service.

In [2]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42
tf.keras.utils.set_random_seed(SEED)

project_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    path for path in project_candidates if (path / "models").is_dir()
)
MODELS_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODELS_DIR / "my_classifier_model.h5"
LABELS_PATH = MODELS_DIR / "labels.json"

CLASS_LABELS = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

print(f"TensorFlow version: {tf.__version__}")
print(f"Model output path: {MODEL_PATH}")

TensorFlow version: 2.21.0
Model output path: c:\Users\shali\ml-prediction-api\models\my_classifier_model.h5


In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"Training images: {x_train.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test images: {x_test.shape}")
print(f"Test labels: {y_test.shape}")

class_counts = np.bincount(y_train, minlength=len(CLASS_LABELS))
for label, count in zip(CLASS_LABELS, class_counts):
    print(f"{label}: {count}")

sample_indices = [np.where(y_train == class_index)[0][0] for class_index in range(len(CLASS_LABELS))]
print(f"Sample image shape: {x_train[sample_indices[0]].shape}")
print(f"Sample labels: {[CLASS_LABELS[y_train[index]] for index in sample_indices]}")

Training images: (50000, 32, 32, 3)
Training labels: (50000,)
Test images: (10000, 32, 32, 3)
Test labels: (10000,)
airplane: 5000
automobile: 5000
bird: 5000
cat: 5000
deer: 5000
dog: 5000
frog: 5000
horse: 5000
ship: 5000
truck: 5000
Sample image shape: (32, 32, 3)
Sample labels: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print(f"Normalized training range: {x_train.min():.1f} to {x_train.max():.1f}")
print(f"Normalized test range: {x_test.min():.1f} to {x_test.max():.1f}")

Normalized training range: 0.0 to 1.0
Normalized test range: 0.0 to 1.0


In [5]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(CLASS_LABELS), activation="softmax"),
    ],
    name="cifar10_classifier",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "cifar10_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 95,434 (372.79 KB)

 Trainable params: 94,986 (371.04 KB)

 Non-trainable params: 448 (1.75 KB)

In [6]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 67ms/step - accuracy: 0.5112 - loss: 1.3666 - val_accuracy: 0.2162 - val_loss: 4.7837
Epoch 2/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 23s 66ms/step - accuracy: 0.6254 - loss: 1.0677 - val_accuracy: 0.6168 - val_loss: 1.0984
Epoch 3/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 23s 64ms/step - accuracy: 0.6707 - loss: 0.9428 - val_accuracy: 0.6084 - val_loss: 1.1031
Epoch 4/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 24s 67ms/step - accuracy: 0.7003 - loss: 0.8595 - val_accuracy: 0.6338 - val_loss: 1.0379
Epoch 5/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 23s 65ms/step - accuracy: 0.7255 - loss: 0.7930 - val_accuracy: 0.6564 - val_loss: 0.9939
Epoch 6/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 26s 72ms/step - accuracy: 0.7431 - loss: 0.7366 - val_accuracy: 0.6900 - val_loss: 0.8903
Epoch 7/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 26s 73ms/step - accuracy: 0.7612 - loss: 0.6904 - val_accuracy: 0.6526 - val_loss: 0.9857
Epoch 8/8
352/352 ━━━━━━━━━━━━━━━━━━━━ 26s 74ms/step - accuracy: 0.7756 - loss: 0.6490 - val_accu

In [7]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

Test loss: 0.9149
Test accuracy: 0.6836
Best validation accuracy: 0.6900


In [8]:
model.save(MODEL_PATH)
LABELS_PATH.write_text(json.dumps(CLASS_LABELS, indent=2), encoding="utf-8")

print(f"Saved model: {MODEL_PATH}")
print(f"Saved labels: {LABELS_PATH}")

Saved model: c:\Users\shali\ml-prediction-api\models\my_classifier_model.h5
Saved labels: c:\Users\shali\ml-prediction-api\models\labels.json


In [9]:
loaded_model = tf.keras.models.load_model(MODEL_PATH)
loaded_labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))

assert MODEL_PATH.exists() and MODEL_PATH.stat().st_size > 0
assert LABELS_PATH.exists() and loaded_labels == CLASS_LABELS
assert loaded_model.output_shape[-1] == len(loaded_labels)

sample_predictions = loaded_model.predict(x_test[:3], verbose=0)
print(f"Model artifact size: {MODEL_PATH.stat().st_size:,} bytes")
print(f"Labels: {loaded_labels}")
print(f"Prediction tensor shape: {sample_predictions.shape}")
print("Model artifacts verified successfully.")

Model artifact size: 1,207,576 bytes
Labels: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Prediction tensor shape: (3, 10)
Model artifacts verified successfully.
